# Zajęcia 11 — obserwowalność i monitoring modeli

Notebook buduje warstwę obserwowalności dla wdrożonego modelu: instrumentację śladami, metryki czterech warstw, SLI/SLO z budżetem błędu, reguły alertów i raport.

Pracujemy na przygotowanym strumieniu logów z doby pracy usługi. Zawiera on trzy okresy: normalny, incydent techniczny oraz okres zmiany rozkładu danych wejściowych.

Komórki opisane jako **zadanie** wypełniasz samodzielnie. Komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`.

## 1. Przygotowanie notebooka

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import json
import sys
import uuid
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE_DIR = REPO_ROOT / "artifacts" / "zajecia_11"
LOGS_DIR = BASE_DIR / "logs"
REPORTS_DIR = BASE_DIR / "reports"
for directory in (LOGS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
FEATURE_COLUMNS = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]
LABELS = ["setosa", "versicolor", "virginica"]

frame = load_iris(as_frame=True).frame
x_train, x_serve, y_train, y_serve = train_test_split(
    frame[FEATURE_COLUMNS], frame["target"], test_size=0.4, random_state=SEED,
    stratify=frame["target"],
)
MODEL = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=500, random_state=SEED)),
    ]
).fit(x_train, y_train)

print(f"Katalog monitoringu: {BASE_DIR.relative_to(REPO_ROOT)}")

%load_ext skip_kernel_extension

## 2. Ślady: gdzie poszedł czas

Metryka mówi, że żądanie trwało 40 ms. Ślad mówi, na co ten czas poszedł. Używamy standardu OpenTelemetry z eksporterem w pamięci — w produkcji zmienia się wyłącznie eksporter, kod instrumentacji zostaje ten sam.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("iris_service")


def handle_request(records: list, request_id: str) -> dict:
    with tracer.start_as_current_span("POST /v1/predict") as root:
        root.set_attribute("request.id", request_id)
        root.set_attribute("request.records", len(records))

        with tracer.start_as_current_span("walidacja_wejscia"):
            batch = pd.DataFrame.from_records(records)[FEATURE_COLUMNS].astype(float)

        with tracer.start_as_current_span("transformacja_cech"):
            transformed = MODEL.named_steps["scaler"].transform(batch)

        with tracer.start_as_current_span("predykcja") as span:
            predictions = MODEL.named_steps["classifier"].predict(transformed)
            span.set_attribute("model.version", "1.0.0")

        return {"request_id": request_id, "predictions": [int(v) for v in predictions]}


response = handle_request(x_serve.head(5).to_dict(orient="records"), str(uuid.uuid4()))
print(f"Odpowiedź: {response}\n")

spans = exporter.get_finished_spans()
display(
    pd.DataFrame(
        [
            {
                "span": span.name,
                "czas_ms": round((span.end_time - span.start_time) / 1e6, 3),
                "atrybuty": dict(span.attributes),
            }
            for span in spans
        ]
    )
)

## 3. Strumień logów z działającej usługi

Generujemy dobę pracy usługi. Log jest **strukturalny** — każdy wpis to jeden obiekt JSON. Zawiera trzy okresy:

- godziny 0–9: praca normalna,
- godziny 10–12: incydent techniczny (wzrost błędów i opóźnienia),
- godziny 16–23: zmiana rozkładu danych wejściowych.

Etykiety pojawiają się z opóźnieniem sześciu godzin — dokładnie tak, jak w większości rzeczywistych problemów. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
rng = np.random.default_rng(SEED)
START = datetime(2026, 3, 2, 0, 0, 0)
NOW = START + timedelta(hours=24)
LABEL_DELAY = timedelta(hours=6)
REQUESTS = 3000

records = []
for index in range(REQUESTS):
    moment = START + timedelta(seconds=index * 86400 / REQUESTS)
    hour = moment.hour
    position = int(rng.integers(0, len(x_serve)))
    features = x_serve.iloc[position].to_dict()
    truth = int(y_serve.iloc[position])

    incident = 10 <= hour < 13
    drift = hour >= 16

    if drift:
        features["petal length (cm)"] = features["petal length (cm)"] + 1.8
        features["petal width (cm)"] = features["petal width (cm)"] + 0.6

    status = 500 if (incident and rng.random() < 0.08) else 200
    latency = float(rng.gamma(shape=4.0, scale=4.0))
    if incident:
        latency *= 4.0

    prediction = None
    if status == 200:
        batch = pd.DataFrame([features])[FEATURE_COLUMNS].astype(float)
        prediction = int(MODEL.predict(batch)[0])

    label_ready = moment + LABEL_DELAY <= NOW

    records.append(
        {
            "timestamp": moment.isoformat(timespec="seconds"),
            "request_id": str(uuid.uuid4()),
            "model_version": "1.0.0",
            "status": status,
            "latency_ms": round(latency, 2),
            "prediction": prediction,
            "label": truth if (label_ready and status == 200) else None,
            "user_email": f"klient{position}@example.com",
            "client_ip": f"10.0.{position % 256}.{index % 256}",
            **{key: round(float(value), 2) for key, value in features.items()},
        }
    )

log_path = LOGS_DIR / "requests.jsonl"
with log_path.open("w", encoding="utf-8") as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

logs = pd.DataFrame(records)
logs["timestamp"] = pd.to_datetime(logs["timestamp"])
logs["hour"] = logs["timestamp"].dt.hour

print(f"Zapisano {len(logs)} wpisów do {log_path.relative_to(REPO_ROOT)}")
print(f"Etykiety dostępne dla {logs['label'].notna().mean():.1%} żądań")
display(logs.head(3))

## 4. Metryki czterech warstw

Liczymy metryki w oknach godzinowych, osobno dla każdej warstwy:

- **system:** udział błędów, opóźnienie p95;
- **dane:** średnia długość płatka oraz udział wartości poza zakresem treningowym;
- **predykcje:** udział najczęstszej klasy;
- **jakość:** dokładność liczona tylko tam, gdzie etykieta jest już znana.

**Zadanie:** napisz funkcję `metrics_by_window(logs)` zwracającą ramkę z jednym wierszem na godzinę i kolumnami dla wszystkich czterech warstw.

In [ ]:
TRAINING_RANGE = {
    column: (float(x_train[column].min()), float(x_train[column].max()))
    for column in FEATURE_COLUMNS
}
print(json.dumps(TRAINING_RANGE, indent=2))

# Zadanie: policz metryki czterech warstw w oknach godzinowych.


In [ ]:
%%skip True
def out_of_range_share(window: pd.DataFrame) -> float:
    flags = []
    for column, (low, high) in TRAINING_RANGE.items():
        flags.append(~window[column].between(low, high))
    return float(pd.concat(flags, axis=1).any(axis=1).mean())


def metrics_by_window(logs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for hour, window in logs.groupby("hour"):
        served = window[window["status"] == 200]
        labelled = served[served["label"].notna()]
        rows.append(
            {
                "hour": int(hour),
                "requests": int(len(window)),
                # warstwa: system
                "error_rate": round(float((window["status"] != 200).mean()), 4),
                "latency_p95_ms": round(float(window["latency_ms"].quantile(0.95)), 2),
                # warstwa: dane
                "petal_length_mean": round(float(window["petal length (cm)"].mean()), 3),
                "out_of_range_share": round(out_of_range_share(window), 4),
                # warstwa: predykcje
                "top_class_share": round(
                    float(served["prediction"].value_counts(normalize=True).max()), 4
                )
                if len(served)
                else np.nan,
                # warstwa: jakosc
                "labelled_share": round(float(len(labelled) / max(len(served), 1)), 4),
                "accuracy": round(float((labelled["prediction"] == labelled["label"]).mean()), 4)
                if len(labelled)
                else np.nan,
            }
        )
    return pd.DataFrame(rows).sort_values("hour").reset_index(drop=True)


windows = metrics_by_window(logs)
windows.to_csv(REPORTS_DIR / "metrics_by_window.csv", index=False)
display(windows)

## 5. SLI, SLO i budżet błędu

Definiujemy dwa wskaźniki liczone **z perspektywy konsumenta**:

- **dostępność:** udział żądań zakończonych kodem 200; cel: 99%;
- **szybkość:** udział żądań obsłużonych poniżej 60 ms; cel: 95%.

**Zadanie:** policz oba wskaźniki dla całej doby, sprawdź spełnienie celu i wyznacz zużycie budżetu błędu. Wynik zapisz do `reports/slo_report.json`.

In [ ]:
SLO = {"dostepnosc": 0.99, "szybkosc": 0.95}
LATENCY_BUDGET_MS = 60.0

# Zadanie: policz SLI, sprawdź SLO i wyznacz zużycie budżetu błędu.


In [ ]:
%%skip True
def slo_report(logs: pd.DataFrame) -> dict:
    availability = float((logs["status"] == 200).mean())
    speed = float((logs["latency_ms"] < LATENCY_BUDGET_MS).mean())

    def budget(actual: float, target: float) -> dict:
        allowed = 1.0 - target
        used = 1.0 - actual
        return {
            "sli": round(actual, 4),
            "slo": target,
            "spelnione": bool(actual >= target),
            "budzet_bledu": round(allowed, 4),
            "zuzyto": round(used, 4),
            "zuzyto_procent_budzetu": round(min(used / allowed, 9.99) * 100, 1),
        }

    report = {
        "okno": "24 h",
        "zadania": int(len(logs)),
        "dostepnosc": budget(availability, SLO["dostepnosc"]),
        "szybkosc": budget(speed, SLO["szybkosc"]),
    }
    (REPORTS_DIR / "slo_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    return report


print(json.dumps(slo_report(logs), indent=2, ensure_ascii=False))

## 6. Reguły alertów

Alert ma sens tylko wtedy, gdy wymaga działania człowieka. Każda reguła składa się z metryki, progu, kierunku i **wymaganej liczby kolejnych okien** — to ostatnie chroni przed pojedynczym wyskokiem.

**Zadanie:** napisz `evaluate_rules(windows, rules)` zwracającą listę alertów: nazwa reguły, okno, wartość, próg. Reguła zapala się dopiero po utrzymaniu przekroczenia przez zadaną liczbę okien.

In [ ]:
RULES = [
    {"nazwa": "wysoki udzial bledow", "metryka": "error_rate", "prog": 0.02, "kierunek": "powyzej", "okna": 2},
    {"nazwa": "wysokie opoznienie p95", "metryka": "latency_p95_ms", "prog": 60.0, "kierunek": "powyzej", "okna": 2},
    {"nazwa": "dane poza zakresem", "metryka": "out_of_range_share", "prog": 0.10, "kierunek": "powyzej", "okna": 2},
    {"nazwa": "spadek jakosci", "metryka": "accuracy", "prog": 0.85, "kierunek": "ponizej", "okna": 1},
]

# Zadanie: oceń reguły alertów na oknach godzinowych.


In [ ]:
%%skip True
def evaluate_rules(windows: pd.DataFrame, rules: list) -> pd.DataFrame:
    alerts = []
    for rule in rules:
        series = windows[rule["metryka"]]
        if rule["kierunek"] == "powyzej":
            breached = series > rule["prog"]
        else:
            breached = series < rule["prog"]
        breached = breached.fillna(False)

        streak = 0
        for position, is_breached in enumerate(breached):
            streak = streak + 1 if is_breached else 0
            if streak == rule["okna"]:
                alerts.append(
                    {
                        "regula": rule["nazwa"],
                        "okno_godzina": int(windows.loc[position, "hour"]),
                        "wartosc": float(series.iloc[position]),
                        "prog": rule["prog"],
                        "kierunek": rule["kierunek"],
                    }
                )
    return pd.DataFrame(alerts)


alerts = evaluate_rules(windows, RULES)
alerts.to_json(REPORTS_DIR / "alerts.json", orient="records", indent=2, force_ascii=False)
display(alerts)

## 7. Punkt kontrolny 1 — wykrycie incydentu

Incydent techniczny zaczął się o godzinie 10. Sprawdzamy, która reguła zadziałała i jak szybko.

**Do opisania:** ile czasu minęło od początku incydentu do jego wykrycia i co trzeba zmienić, żeby skrócić ten czas o połowę?

In [ ]:
INCIDENT_START_HOUR = 10

technical = alerts[alerts["regula"].isin(["wysoki udzial bledow", "wysokie opoznienie p95"])]
display(technical)

if not technical.empty:
    first = int(technical["okno_godzina"].min())
    print(f"Początek incydentu:   godzina {INCIDENT_START_HOUR}")
    print(f"Pierwszy alert:       godzina {first}")
    print(f"Czas do wykrycia:     {first - INCIDENT_START_HOUR} h")
    print("\nCzas wykrycia wynika z długości okna i wymaganej liczby kolejnych okien.")
else:
    print("UWAGA: żadna reguła techniczna nie zadziałała — progi są za wysokie.")

## 8. Punkt kontrolny 2 — zbyt czuły próg

Porównujemy tę samą regułę w dwóch wariantach: rozsądnym i przeczulonym.

**Do opisania:** ile fałszywych alarmów generuje wersja czuła i co się stanie z zespołem, który dostaje je codziennie?

In [ ]:
def count_alerts(threshold: float, windows_required: int) -> pd.DataFrame:
    rule = [
        {
            "nazwa": f"bledy > {threshold}",
            "metryka": "error_rate",
            "prog": threshold,
            "kierunek": "powyzej",
            "okna": windows_required,
        }
    ]
    return evaluate_rules(windows, rule)


sensible = count_alerts(0.02, 2)
sensitive = count_alerts(0.001, 1)

print(f"Próg 2%, wymagane 2 okna:    {len(sensible)} alert(ów)")
print(f"Próg 0,1%, wymagane 1 okno:  {len(sensitive)} alert(ów)")
print(f"\nOkna z niezerowym udziałem błędów: {int((windows['error_rate'] > 0).sum())} z {len(windows)}")
display(sensitive.head(10))

## 9. Punkt kontrolny 3 — dane wrażliwe w logach

Log usługi predykcyjnej jest najczęstszym miejscem niezamierzonego wycieku danych.

**Zadanie:** znajdź w logu pola, których nie powinno tam być, i napisz funkcję `redact(record)` usuwającą albo maskującą je, bez utraty możliwości diagnozy pojedynczego żądania.

In [ ]:
print("Pola w logu:")
print(json.dumps(records[0], indent=2, ensure_ascii=False))

# Zadanie: wskaż pola wrażliwe i napisz funkcję maskującą.


In [ ]:
%%skip True
import hashlib
import re

SENSITIVE_PATTERNS = {
    "adres e-mail": re.compile(r"[^@\s]+@[^@\s]+\.[a-z]{2,}"),
    "adres IP": re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b"),
}


def find_sensitive_fields(record: dict) -> list:
    found = []
    for key, value in record.items():
        for label, pattern in SENSITIVE_PATTERNS.items():
            if isinstance(value, str) and pattern.fullmatch(value):
                found.append({"pole": key, "typ": label})
    return found


def redact(record: dict) -> dict:
    clean = dict(record)
    # Adres e-mail zastępujemy pseudonimem — nadal łączy zdarzenia tego samego klienta,
    # ale nie pozwala go zidentyfikować.
    if "user_email" in clean:
        digest = hashlib.sha256(clean.pop("user_email").encode("utf-8")).hexdigest()
        clean["client_pseudonym"] = digest[:12]
    # Adres IP nie jest potrzebny do diagnozy predykcji.
    clean.pop("client_ip", None)
    return clean


print("Znalezione pola wrażliwe:")
display(pd.DataFrame(find_sensitive_fields(records[0])))

print("\nWpis po zamaskowaniu:")
print(json.dumps(redact(records[0]), indent=2, ensure_ascii=False))

safe_path = LOGS_DIR / "requests_redacted.jsonl"
with safe_path.open("w", encoding="utf-8") as handle:
    for record in records:
        handle.write(json.dumps(redact(record), ensure_ascii=False) + "\n")
print(f"\nZapisano bezpieczną wersję logu: {safe_path.name}")
print("Retencja: log diagnostyczny 30 dni, statystyki metryk bezterminowo.")

## 10. Raport monitoringu

Dashboard nie jest wymagany — cykliczny raport w repozytorium daje ten sam efekt przy zerowej infrastrukturze.

**Zadanie:** wygeneruj `reports/monitoring_report.md` zawierający: podsumowanie SLO, listę alertów, tabelę metryk godzinowych oraz wskazanie okien, w których jakość spadła.

In [ ]:
# Zadanie: wygeneruj raport monitoringu.


In [ ]:
%%skip True
def to_markdown_table(frame: pd.DataFrame) -> str:
    columns = list(frame.columns)
    header = "| " + " | ".join(columns) + " |"
    separator = "|" + "|".join(["---"] * len(columns)) + "|"
    body = [
        "| " + " | ".join("" if pd.isna(row[column]) else str(row[column]) for column in columns) + " |"
        for _, row in frame.iterrows()
    ]
    return "\n".join([header, separator, *body])


slo = json.loads((REPORTS_DIR / "slo_report.json").read_text(encoding="utf-8"))
weak = windows[windows["accuracy"] < 0.85][["hour", "accuracy", "out_of_range_share"]]

report = f"""# Raport monitoringu — usługa iris_classifier

Okno: doba, {slo['zadania']} żądań.

## Poziom usługi

| Wskaźnik | SLI | SLO | Spełnione | Zużycie budżetu błędu |
|---|---|---|---|---|
| Dostępność | {slo['dostepnosc']['sli']} | {slo['dostepnosc']['slo']} | {slo['dostepnosc']['spelnione']} | {slo['dostepnosc']['zuzyto_procent_budzetu']}% |
| Szybkość | {slo['szybkosc']['sli']} | {slo['szybkosc']['slo']} | {slo['szybkosc']['spelnione']} | {slo['szybkosc']['zuzyto_procent_budzetu']}% |

## Alerty

{to_markdown_table(alerts) if not alerts.empty else 'Brak alertów.'}

## Okna z obniżoną jakością

{to_markdown_table(weak) if not weak.empty else 'Brak.'}

## Metryki godzinowe

{to_markdown_table(windows)}

## Uwagi

- Etykiety pojawiają się z opóźnieniem 6 godzin, więc metryka jakości opisuje przeszłość.
- W oknach bez etykiet obserwujemy rozkład wejścia i rozkład predykcji.
- Log diagnostyczny przechowujemy w wersji zamaskowanej, z retencją 30 dni.
"""

(REPORTS_DIR / "monitoring_report.md").write_text(report, encoding="utf-8")
print(report[:1500])

## 11. Metryki, które wystawia usługa

Wszystko, co liczyliśmy dotąd, było liczone **po fakcie**, z pliku logów. W produkcji działa to odwrotnie: usługa **wystawia** swój stan pod adresem `/metrics`, a system monitorujący sam po niego przychodzi w regularnych odstępach.

Ten model nazywa się *pull* i ma trzy konsekwencje, które warto rozumieć:

- usługa nie musi wiedzieć, kto ją obserwuje, ani co się dzieje, gdy monitoring przestanie działać;
- metryki są **agregatami w pamięci procesu**, a nie zapisem każdego zdarzenia — dlatego są tanie;
- restart procesu zeruje liczniki, więc pytania zadaje się o **przyrost** (`rate`), a nie o wartość bezwzględną.

Biblioteka `prometheus_client` daje trzy typy, które wystarczają na początek:

| Typ | Do czego | Przykład |
|---|---|---|
| `Counter` | wartości, które tylko rosną | liczba predykcji, liczba odrzuconych żądań |
| `Histogram` | rozkład wartości w koszykach | czas obsługi, pewność predykcji |
| `Gauge` | wartość, która może rosnąć i maleć | żądania w toku, wersja modelu |

**Zadanie:** napisz usługę wystawiającą `/metrics` z metrykami **czterech warstw** z sekcji 4: usługi, danych, modelu i biznesu. Uruchom ją, wyślij ruch i odczytaj metryki tak, jak zrobiłby to system monitorujący.

In [ ]:
# Zadanie: napisz usługę eksportującą metryki i odczytaj je z /metrics.


In [ ]:
%%skip True
import socket
import subprocess
import textwrap
import time

import httpx
import joblib

SERVICE_DIR = BASE_DIR / "usluga"
SERVICE_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(MODEL, SERVICE_DIR / "model.joblib")

(SERVICE_DIR / "app.py").write_text(
    textwrap.dedent(
        """
        # Usluga predykcyjna wystawiajaca wlasny stan pod /metrics.
        import os
        import time
        from pathlib import Path

        import joblib
        import numpy as np
        from fastapi import FastAPI, Response
        from prometheus_client import CONTENT_TYPE_LATEST, Counter, Gauge, Histogram, generate_latest
        from pydantic import BaseModel, Field, ValidationError

        MODEL = joblib.load(Path(__file__).parent / 'model.joblib')
        LABELS = ['setosa', 'versicolor', 'virginica']
        FEATURES = [
            'sepal length (cm)',
            'sepal width (cm)',
            'petal length (cm)',
            'petal width (cm)',
        ]

        # warstwa uslugi
        REQUESTS = Counter('http_requests_total', 'Liczba zadan', ['endpoint', 'status'])
        LATENCY = Histogram(
            'prediction_latency_seconds',
            'Czas obslugi zadania predykcji',
            buckets=(0.001, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0),
        )
        IN_FLIGHT = Gauge('requests_in_flight', 'Zadania w toku')

        # warstwa danych
        REJECTED = Counter('input_rejected_total', 'Zadania odrzucone przez kontrakt', ['powod'])
        FEATURE_VALUES = Histogram(
            'feature_value',
            'Rozklad wartosci cech wejsciowych',
            ['cecha'],
            buckets=(0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0),
        )

        # warstwa modelu
        PREDICTIONS = Counter('predictions_total', 'Liczba predykcji', ['klasa'])
        CONFIDENCE = Histogram(
            'prediction_confidence',
            'Pewnosc predykcji',
            buckets=(0.34, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0),
        )

        # warstwa biznesowa: predykcje niepewne, ktore trafiaja do weryfikacji przez czlowieka
        ESCALATED = Counter('escalated_to_human_total', 'Predykcje skierowane do weryfikacji')
        CONFIDENCE_THRESHOLD = float(os.environ.get('CONFIDENCE_THRESHOLD', '0.9'))

        MODEL_INFO = Gauge('model_info', 'Wersja wdrozonego modelu', ['wersja'])
        MODEL_INFO.labels(wersja=os.environ.get('MODEL_VERSION', '1.0.0')).set(1)

        app = FastAPI(title='Iris service z metrykami')


        class IrisRecord(BaseModel):
            sepal_length_cm: float = Field(ge=0.0, le=20.0)
            sepal_width_cm: float = Field(ge=0.0, le=20.0)
            petal_length_cm: float = Field(ge=0.0, le=20.0)
            petal_width_cm: float = Field(ge=0.0, le=20.0)


        @app.get('/health')
        def health() -> dict:
            REQUESTS.labels(endpoint='/health', status='200').inc()
            return {'status': 'alive'}


        @app.get('/metrics')
        def metrics() -> Response:
            return Response(generate_latest(), media_type=CONTENT_TYPE_LATEST)


        @app.post('/v1/predict')
        def predict(payload: dict, response: Response) -> dict:
            IN_FLIGHT.inc()
            started = time.perf_counter()
            try:
                try:
                    record = IrisRecord(**payload)
                except ValidationError as error:
                    REJECTED.labels(powod=error.errors()[0]['type']).inc()
                    REQUESTS.labels(endpoint='/v1/predict', status='422').inc()
                    response.status_code = 422
                    return {'error': 'kontrakt wejscia naruszony'}

                values = [float(value) for value in record.model_dump().values()]
                for name, value in zip(FEATURES, values):
                    FEATURE_VALUES.labels(cecha=name).observe(value)

                probabilities = MODEL.predict_proba(np.array([values]))[0]
                index = int(np.argmax(probabilities))
                confidence = float(probabilities[index])

                PREDICTIONS.labels(klasa=LABELS[index]).inc()
                CONFIDENCE.observe(confidence)
                escalated = confidence < CONFIDENCE_THRESHOLD
                if escalated:
                    ESCALATED.inc()

                REQUESTS.labels(endpoint='/v1/predict', status='200').inc()
                return {
                    'label': LABELS[index],
                    'confidence': round(confidence, 4),
                    'escalated': escalated,
                }
            finally:
                LATENCY.observe(time.perf_counter() - started)
                IN_FLIGHT.dec()
        """
    ).lstrip(),
    encoding="utf-8",
)


def free_port() -> int:
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        return probe.getsockname()[1]


PORT = free_port()
service = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=SERVICE_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
BASE_URL = f"http://127.0.0.1:{PORT}"
FIELDS = ["sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"]

try:
    for _ in range(60):
        try:
            if httpx.get(f"{BASE_URL}/health", timeout=1.0).status_code == 200:
                break
        except Exception:
            time.sleep(0.5)
    else:
        raise RuntimeError("Usługa nie odpowiedziała w wyznaczonym czasie.")

    for _, row in x_serve.iterrows():
        payload = dict(zip(FIELDS, [float(value) for value in row]))
        httpx.post(f"{BASE_URL}/v1/predict", json=payload, timeout=5.0)

    # Dwa zadania lamiace kontrakt wejscia: brak cechy i wartosc poza zakresem.
    for bad in ({"sepal_length_cm": 5.1}, dict(zip(FIELDS, [999.0, 3.5, 1.4, 0.2]))):
        httpx.post(f"{BASE_URL}/v1/predict", json=bad, timeout=5.0)

    exposition = httpx.get(f"{BASE_URL}/metrics", timeout=5.0).text
finally:
    service.terminate()
    service.wait(timeout=10)

(REPORTS_DIR / "metrics_exposition.txt").write_text(exposition, encoding="utf-8")

INTERESTING = (
    "predictions_total",
    "input_rejected_total",
    "escalated_to_human_total",
    "prediction_latency_seconds_count",
    "prediction_latency_seconds_sum",
    "model_info",
)
print("Odczyt spod /metrics — tak widzi usługę system monitorujący:\n")
for line in exposition.splitlines():
    if line.startswith(INTERESTING):
        print("  " + line)

## 12. Stos obserwowalności

Usługa wystawia liczby. Do pełnego obrazu potrzebne są jeszcze trzy elementy, każdy odpowiadający na inne pytanie:

| Składnik | Pytanie, na które odpowiada |
|---|---|
| **Prometheus** | jak metryki zmieniały się w czasie? |
| **Grafana** | jak to wygląda i kiedy przekroczyliśmy próg? |
| **Jaeger** | gdzie dokładnie poszedł czas w tym jednym wolnym żądaniu? |

Dwie rzeczy warte uwagi w konfiguracji poniżej:

- Prometheus **sam przychodzi** po metryki pod adres usługi, więc w konfiguracji podajemy jej adres, a nie odwrotnie;
- hasło do Grafany pochodzi ze zmiennej środowiskowej i **nie ma wartości domyślnej** — `docker compose` odmówi startu, dopóki jej nie ustawisz. Domyślne hasło administratora zapisane w pliku wersjonowanym w repozytorium to jeden z częstszych sposobów wystawienia panelu na świat.

**Zadanie:** zapisz `compose.yaml`, `prometheus.yml` oraz konfigurację źródła danych dla Grafany.

In [ ]:
# Zadanie: opisz stos obserwowalności.


In [ ]:
%%skip True
import shutil

STACK_DIR = BASE_DIR / "stos"
(STACK_DIR / "grafana").mkdir(parents=True, exist_ok=True)


def write_stack(relative_path: str, content: str) -> Path:
    path = STACK_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    print(f"Zapisano: stos/{relative_path}")
    return path


write_stack(
    "compose.yaml",
    """
    # Uruchomienie (PowerShell):
    #   $env:GRAFANA_PASSWORD = '<wlasne haslo>'
    #   docker compose up -d
    services:
      prometheus:
        image: prom/prometheus:v3.1.0
        command:
          - --config.file=/etc/prometheus/prometheus.yml
          - --storage.tsdb.retention.time=6h
        volumes:
          - type: bind
            source: ./prometheus.yml
            target: /etc/prometheus/prometheus.yml
            read_only: true
        ports:
          - '9090:9090'

      grafana:
        image: grafana/grafana:11.4.0
        environment:
          # Brak wartosci domyslnej jest celowy - compose odmowi startu bez tej zmiennej.
          GF_SECURITY_ADMIN_PASSWORD: ${GRAFANA_PASSWORD:?ustaw zmienna GRAFANA_PASSWORD}
          GF_AUTH_ANONYMOUS_ENABLED: 'false'
          GF_USERS_ALLOW_SIGN_UP: 'false'
        volumes:
          - type: bind
            source: ./grafana/datasources.yaml
            target: /etc/grafana/provisioning/datasources/datasources.yaml
            read_only: true
        ports:
          - '3000:3000'
        depends_on:
          - prometheus

      jaeger:
        image: jaegertracing/all-in-one:1.65
        environment:
          COLLECTOR_OTLP_ENABLED: 'true'
        ports:
          - '16686:16686'   # panel
          - '4318:4318'     # OTLP po HTTP
    """,
)

write_stack(
    "prometheus.yml",
    """
    # Prometheus sam przychodzi po metryki - to usluga jest celem, nie nadawca.
    global:
      scrape_interval: 5s
      evaluation_interval: 15s

    scrape_configs:
      - job_name: iris-service
        static_configs:
          # host.docker.internal wskazuje maszyne, na ktorej dziala Docker.
          - targets: ['host.docker.internal:8009']
            labels:
              srodowisko: staging

      - job_name: prometheus
        static_configs:
          - targets: ['localhost:9090']
    """,
)

write_stack(
    "grafana/datasources.yaml",
    """
    apiVersion: 1

    datasources:
      - name: Prometheus
        type: prometheus
        access: proxy
        url: http://prometheus:9090
        isDefault: true
    """,
)

write_stack(
    "zapytania.md",
    """
    # Zapytania do wklejenia w Grafanie

    Liczniki tylko rosna i zeruja sie przy restarcie, wiec pytamy o **przyrost**:

    ```promql
    # przepustowosc: predykcje na sekunde, w podziale na klasy
    rate(predictions_total[1m])

    # udzial zadan odrzuconych przez kontrakt wejscia
    rate(input_rejected_total[5m]) / rate(http_requests_total{endpoint='/v1/predict'}[5m])

    # opoznienie p95 liczone z histogramu
    histogram_quantile(0.95, rate(prediction_latency_seconds_bucket[5m]))

    # udzial predykcji skierowanych do czlowieka - metryka warstwy biznesowej
    rate(escalated_to_human_total[15m]) / rate(predictions_total[15m])

    # rozklad klas: sygnal dryfu widoczny bez etykiet
    sum by (klasa) (rate(predictions_total[30m]))
    ```

    Ostatnie zapytanie jest najciekawsze: nagla zmiana proporcji klas jest
    sygnalem dryfu, ktory widac **bez znajomosci prawdziwych etykiet**.
    Do tego wracamy na zajeciach 12.
    """,
)

# Uruchomienie stosu pobiera kilkaset megabajtow obrazow, wiec jest opcjonalne.
URUCHOM_STOS = False

if not URUCHOM_STOS:
    print("\n[DOCKER] Uruchomienie stosu wyłączone (URUCHOM_STOS = False).")
    print("Aby go uruchomić, ustaw URUCHOM_STOS = True albo wykonaj w katalogu stos/:")
    print("  docker compose up -d")
    print("  Prometheus :9090 | Grafana :3000 | Jaeger :16686")
elif shutil.which("docker") is None:
    print("\n[DOCKER] Brak polecenia docker — pomijam uruchomienie stosu.")
else:
    launched = subprocess.run(
        ["docker", "compose", "up", "-d"], cwd=STACK_DIR, capture_output=True, text=True
    )
    print((launched.stdout + launched.stderr).strip()[-1200:])

## 13. Poza notebookiem — praca na stosie

### Krok 1 — uruchomienie

```powershell
$env:GRAFANA_PASSWORD = '<wlasne haslo>'
cd artifacts/zajecia_11/stos
docker compose up -d
```

Pierwsze uruchomienie pobiera obrazy i trwa kilka minut.

### Krok 2 — usługa jako cel

Uruchom usługę z sekcji 11 na porcie zgodnym z `prometheus.yml`:

```bash
cd artifacts/zajecia_11/usluga
python -m uvicorn app:app --host 0.0.0.0 --port 8009
```

W Prometheusie sprawdź **Status → Targets**. Cel `iris-service` musi mieć stan `UP`. Jeżeli ma `DOWN`, problemem jest zwykle adres `host.docker.internal` albo zapora — to typowe pierwsze potknięcie i warto je zobaczyć.

### Krok 3 — wykresy

Zaloguj się do Grafany (`admin` i hasło z `GRAFANA_PASSWORD`), utwórz panel i wklej zapytania z pliku `zapytania.md`. Wyślij ruch do usługi i obserwuj, jak wykresy reagują.

### Krok 4 — ślady

Ustaw przed startem usługi:

```bash
export OTEL_EXPORTER_OTLP_ENDPOINT=http://localhost:4318
export OTEL_SERVICE_NAME=iris-service
opentelemetry-instrument python -m uvicorn app:app --port 8009
```

Ślady pojawią się w Jaegerze pod adresem `http://localhost:16686`. Znajdź najwolniejsze żądanie i sprawdź, w którym fragmencie obsługi spędziło czas.

### Krok 5 — sprzątanie

```bash
docker compose down -v
```

## 14. Podsumowanie

System przestał być nieprzezroczysty. Wiemy, jak zachowuje się usługa, jakie dane do niej trafiają, co przewiduje model i jaka jest jego jakość tam, gdzie znamy już odpowiedź. Alerty mają progi i okna, a raport powstaje automatycznie.

Czego nadal brakuje:

- widzimy, że coś się zmieniło w danych, ale nie mierzymy tego formalnie ani nie oceniamy istotności (zajęcia 12);
- nie potrafimy oszacować jakości modelu w oknie bez etykiet (zajęcia 12);
- wykrycie problemu nie uruchamia żadnej reakcji — ponownego treningu ani porównania z challengerem (zajęcia 12).

### Zadanie do własnego projektu

1. Instrumentacja: identyfikator żądania, log strukturalny, co najmniej jeden ślad.
2. Metryki na czterech warstwach — po co najmniej jednej z każdej.
3. Opis momentu pojawiania się etykiet i tego, co monitorujesz w okresie ich braku.
4. Jedno SLI i SLO wraz z budżetem błędu.
5. Co najmniej dwie reguły alertów z uzasadnieniem progów i okien.
6. Raport monitoringu jako artefakt w repozytorium.
7. Przegląd logów pod kątem danych wrażliwych i przyjęta retencja.